# Vérification du téléchargement BD TOPO

Ce notebook vérifie que `download.bd_topo.download_bd_topo` interroge correctement le WFS de la Géoplateforme IGN sur l'emprise d'une entité du CSV, écrit les couches en GeoPackage dans `data/vector/bd_topo/`, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.bd_topo import download_bd_topo
from download.limites_administratives import fetch_commune_boundary, resolve_bbox

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des couches BD TOPO sur la bbox de l'entité

In [ ]:
bbox = resolve_bbox(entity)
written = download_bd_topo(bbox)
written

## 3. Relecture des GeoPackage écrits

In [ ]:
import geopandas as gpd

layers = {name: gpd.read_file(path) for name, path in written.items()}
for name, gdf in layers.items():
    print(name, "->", len(gdf), "entites")

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

styles = {
    "batiment": {"color": "orange", "fillColor": "orange", "fillOpacity": 0.5},
    "troncon_de_route": {"color": "gray", "weight": 2},
    "route_numerotee_ou_nommee": {"color": "black", "weight": 3},
    "voie_nommee": {"color": "dimgray", "weight": 1},
    "itineraire_autre": {"color": "sienna", "weight": 1},
    "point_d_acces": {"color": "crimson", "fillColor": "crimson", "fillOpacity": 0.6},
    "point_de_repere": {"color": "darkorange", "fillColor": "darkorange", "fillOpacity": 0.6},
    "section_de_points_de_repere": {"color": "orange", "weight": 2},
    "non_communication": {"color": "red", "fillColor": "red", "fillOpacity": 0.6},
    "equipement_de_transport": {"color": "purple", "fillColor": "purple", "fillOpacity": 0.6},
    "troncon_hydrographique": {"color": "blue", "weight": 2},
    "cours_d_eau": {"color": "royalblue", "weight": 2},
    "plan_d_eau": {"color": "dodgerblue", "fillColor": "dodgerblue", "fillOpacity": 0.4},
    "surface_hydrographique": {"color": "deepskyblue", "fillColor": "deepskyblue", "fillOpacity": 0.4},
    "detail_hydrographique": {"color": "navy", "fillColor": "navy", "fillOpacity": 0.6},
    "noeud_hydrographique": {"color": "darkblue", "fillColor": "darkblue", "fillOpacity": 0.6},
    "limite_terre_mer": {"color": "teal", "weight": 2},
    "zone_d_estran": {"color": "cadetblue", "fillColor": "cadetblue", "fillOpacity": 0.3},
    "haie": {"color": "darkgreen", "weight": 2},
    "zone_de_vegetation": {"color": "green", "fillColor": "green", "fillOpacity": 0.3},
    "foret_publique": {"color": "forestgreen", "fillColor": "forestgreen", "fillOpacity": 0.3},
    "zone_d_habitation": {"color": "gold", "fillColor": "gold", "fillOpacity": 0.2},
    "cimetiere": {"color": "slategray", "fillColor": "slategray", "fillOpacity": 0.4},
    "parc_ou_reserve": {"color": "lightgreen", "fillColor": "lightgreen", "fillOpacity": 0.3},
    "zone_d_activite_ou_d_interet": {"color": "brown", "fillColor": "brown", "fillOpacity": 0.3},
    "terrain_de_sport": {"color": "lime", "fillColor": "lime", "fillOpacity": 0.3},
}

boundary = fetch_commune_boundary(entity.code_insee)
boundary_wgs84 = boundary.to_crs(epsg=4326)

m = leafmap.Map()

last_gdf_wgs84 = None
for name, gdf in layers.items():
    gdf_wgs84 = gdf.to_crs(epsg=4326)
    last_gdf_wgs84 = gdf_wgs84
    m.add_gdf(
        gdf_wgs84,
        layer_name=name,
        style=styles.get(name),
        info_mode="on_click",
    )

m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)

if last_gdf_wgs84 is not None:
    m.zoom_to_gdf(last_gdf_wgs84)

m